<a href="https://colab.research.google.com/github/elkins/torsion-tuner/blob/main/examples/multi_modal_refinement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Modal GNN Refinement

This tutorial walks through `TorsionTuner`, a Graph Neural Network (GNN) pipeline that refines protein backbone coordinates by optimizing torsion angles against multiple experimental data sources simultaneously (SAXS, NMR Chemical Shifts, and Ramachandran geometric priors).

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Requires diff-biophys for differentiable SAXS/NMR models
    !pip install -q git+https://github.com/elkins/diff-biophys.git
    !pip install -q TorsionTuner biotite matplotlib jax jaxlib equinox optax
else:
    import os
    sys.path.insert(0, os.path.abspath("../"))

## 1. Load the Target Data
In a real scenario, you would have experimental SAXS $I(q)$ curves and NMR chemical shifts. Here, we mock this data generation from a known ground truth structure to see if our GNN can recover it.

In [ ]:
import jax.numpy as jnp
from diff_biophys.saxs import debye_saxs
from diff_biophys.nmr.chemical_shifts import predict_ca_shifts
from torsiontuner.data import load_pdb, get_graph_features
from torsiontuner.montelione_utils import get_residue_rc_shifts

# We use the provided test helix
data = load_pdb("../test_helix.pdb")
node_features, adj, edge_features = get_graph_features(data)

# Generate mock SAXS target
q_values = jnp.linspace(0.01, 0.5, 50)
form_factors = jnp.ones((1, len(q_values))) * 6.0  # Simplified carbon scattering
target_saxs = debye_saxs(data["coords"], q_values, form_factors)

# Generate mock NMR target
true_phi = data["dihedrals"][2::3]
true_psi = data["dihedrals"][0::3]
rc_shifts = get_residue_rc_shifts(data["res_indices"])
target_shifts = predict_ca_shifts(true_phi, true_psi, rc_shifts[1:])

print("Mock experimental targets successfully generated.")

## 2. Initialize the GNN and Optimizer
We use `equinox` to define the GNN and `optax` for AdamW optimization.

In [ ]:
import jax.random as jr
import equinox as eqx
import optax
from torsiontuner.model import FineTunerGNN

key = jr.PRNGKey(42)
model = FineTunerGNN(
    node_dim=20,
    hidden_dim=64,
    out_dim=2, # Outputs delta_phi, delta_psi
    n_layers=3,
    key=key,
)

optimizer = optax.adamw(learning_rate=5e-3)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

## 3. The Multi-Objective Loss Function
The loss function is a weighted sum of four terms:
1. `saxs_loss`: Mean squared error against $I(q)$
2. `nmr_loss`: Error against C-alpha chemical shifts
3. `rama_loss`: Penalty for torsion angles falling outside allowed Ramachandran regions
4. `reg_loss`: L2 regularization to prevent massive unfolding steps

In [ ]:
from torsiontuner.kinematics import rebuild_backbone
from torsiontuner.montelione_utils import montelione_loss, ramachandran_penalty
import jax

def loss_fn(model, *args):
    # Predict torsional adjustments
    deltas = model(node_features, adj, edge_features)

    # Reconstruct 3D Cartesian coordinates from new torsions
    refined_coords, updated_dihedrals = rebuild_backbone(
        data["init_coords"], data["lengths"], data["angles"], data["dihedrals"], deltas
    )

    # 1. SAXS Loss
    pred_saxs = debye_saxs(refined_coords, q_values, form_factors)
    saxs_loss = jnp.mean(((pred_saxs - target_saxs) / jnp.max(target_saxs)) ** 2)

    # 2. NMR Loss
    pred_phi = updated_dihedrals[2::3]
    pred_psi = updated_dihedrals[0::3]
    nmr_loss = montelione_loss(pred_phi, pred_psi, target_shifts, data["res_indices"][1:])

    # 3. Ramachandran Loss
    rama_loss = ramachandran_penalty(pred_phi, pred_psi)

    # Total
    return saxs_loss + 0.1 * nmr_loss + 0.05 * rama_loss + 0.01 * jnp.mean(deltas**2)

@eqx.filter_jit
def make_step(model, opt_state):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

## 4. Run Refinement Loop

In [ ]:
import matplotlib.pyplot as plt

losses = []
print("Refining structure...")
for step in range(50):
    model, opt_state, loss = make_step(model, opt_state)
    losses.append(loss)

plt.figure(figsize=(6, 4))
plt.plot(losses, color='darkred', linewidth=2)
plt.title("Multi-Objective Refinement Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.show()

print("Refinement complete. Coordinates are now optimally fitted to SAXS and NMR data.")